# 🚀 Sistema RAG Completo con Groq (GRATIS y Rápido)
**Arquitectura explicada paso a paso**

**Pasos RAG:**
1. **Carga** → Lee documentos
2. **Divide** → Chunks manejables
3. **Incrusta** → Texto → vectores
4. **Almacena** → Vector DB
5. **Recupera** → Chunks relevantes
6. **Genera** → LLM + contexto

## 1. INSTALACIÓN Y CONFIGURACIÓN

In [1]:
!pip install tqdm -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from tqdm import tqdm
import subprocess, sys

# Lista de paquetes necesarios para el proyecto:
# - langchain-classic/groq/chroma/community/huggingface → componentes del pipeline RAG
# - faiss-cpu          → vector store en memoria, búsqueda por similitud coseno
# - sentence-transformers → modelo de embeddings multilingüe local (sin API)
# - chromadb           → vector store persistente en disco
# - cohere             → reranker: reordena los chunks recuperados por relevancia
# - pymupdf            → carga y extracción de texto de PDFs
# - beautifulsoup4     → parseado de HTML
packages = [
    "langchain-classic", "langchain-groq", "langchain-chroma",
    "faiss-cpu", "python-dotenv", "langchain-huggingface",
    "beautifulsoup4", "sentence-transformers", "chromadb",
    "langchain-cohere", "pymupdf", "langchain_community"
]

errores = []
for package in tqdm(packages, desc='📦 Instalando paquetes'):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', package, '-q'],
        capture_output=True
    )
    if result.returncode != 0:
        errores.append(package)

if errores:
    print(f"❌ Fallaron: {errores}")
else:
    print("✅ Todos los paquetes instalados correctamente")


📦 Instalando paquetes: 100%|██████████| 12/12 [00:18<00:00,  1.55s/it]

✅ Todos los paquetes instalados correctamente


In [39]:
from dotenv import load_dotenv

# Crea un archivo .env en la raíz del proyecto con tu API key:
#   GROQ_API_KEY=gsk_xxxxxxxxxxxx
# Obtén una clave gratuita en: https://console.groq.com/keys
# Para LangSmith (evaluación), añade también:
#   LANGSMITH_API_KEY=lsv2_xxxxxxxxxxxx  → https://smith.langchain.com
#   LANGSMITH_PROJECT=mi-rag-groq
load_dotenv()
print('✅ .env cargado')

✅ .env cargado


## 2. CARGA DE DOCUMENTOS

In [40]:
# ─── IMPORTS ──────────────────────────────────────────────────────────────────
import json, re, os
from langchain_community.document_loaders import WebBaseLoader, PyMuPDFLoader

# ─── USER AGENT ───────────────────────────────────────────────────────────────
# WebBaseLoader hace peticiones HTTP simples. Por defecto se identifica como un bot,
# lo que puede provocar que algunos servidores devuelvan contenido reducido o bloqueen
# la petición. Configurando un User-Agent de Chrome real nos identificamos como un
# navegador normal, reduciendo la probabilidad de bloqueos.
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# ─── FUNCIONES DE LIMPIEZA ────────────────────────────────────────────────────
# Aunque WebBaseLoader ya extrae el texto, el HTML de las webs genera artefactos
# como saltos de línea múltiples y espacios extra que ensucian los chunks.
def clean_web_doc(doc):
    text = doc.page_content
    text = re.sub(r'\n{3,}', '\n\n', text)      # colapsa saltos de línea múltiples
    text = re.sub(r'[ \t]{2,}', ' ', text)       # colapsa espacios y tabulaciones
    doc.page_content = text.strip()
    return doc

# PyMuPDF genera artefactos específicos del formato PDF:
# palabras cortadas con guión al final de línea, referencias internas a imágenes
# (cid:N), caracteres de fuentes embebidas no-ASCII, y etiquetas HTML residuales
# de PDFs generados desde Word o HTML.
def clean_pdf_doc(doc):
    text = doc.page_content
    text = re.sub(r'-\n', '', text)              # une palabras cortadas por salto de página
    text = re.sub(r'\n{3,}', '\n\n', text)       # colapsa saltos múltiples
    text = re.sub(r'[ \t]{2,}', ' ', text)       # colapsa espacios y tabulaciones
    text = re.sub(r'<[^>]+>', '', text)           # elimina etiquetas HTML residuales
    text = re.sub(r'\(cid:\d+\)', '', text)       # elimina referencias internas de PyMuPDF a imágenes/fuentes
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)   # elimina caracteres no ASCII (iconos, símbolos de fuentes embebidas)
    doc.page_content = text.strip()
    return doc

# ─── CARGA WEB ────────────────────────────────────────────────────────────────
# WebBaseLoader hace una petición HTTP simple y extrae el texto con BeautifulSoup.
# Funciona bien con webs que renderizan el HTML en servidor (server-side rendering),
# como la mayoría de periódicos, porque necesitan que Google (también un bot) indexe su contenido.
# ⚠️ No funciona con SPAs (React/Angular/Vue) que generan el contenido con JavaScript
#    en el navegador, ni con webs que bloquean scrapers con CDNs como Akamai.
#    Para esos casos usar Selenium o Playwright con Python 3.11.
with open("urls.json", "r", encoding="utf-8") as f:
    data = json.load(f)

loader = WebBaseLoader(data["news"])
url_docs = loader.load()
url_docs = [clean_web_doc(d) for d in url_docs]
for doc in url_docs:
    doc.metadata["source_type"] = "web"
print(f"📄 Cargados {len(url_docs)} documentos web")

# ─── CARGA PDF ────────────────────────────────────────────────────────────────
# PyMuPDFLoader extrae el texto página a página. Cada página se convierte en un
# Document independiente, por eso un PDF de 4 páginas genera 4 documentos.
pdf_loader = PyMuPDFLoader("organic.pdf")
pdf_docs = pdf_loader.load()
pdf_docs = [clean_pdf_doc(d) for d in pdf_docs]
for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"
print(f"📄 Cargados {len(pdf_docs)} documentos PDF")

# ─── MERGE ────────────────────────────────────────────────────────────────────
# Combinamos web + PDF en una sola lista. A partir de aquí el pipeline
# trata todos los documentos igual independientemente de su origen.
docs = url_docs + pdf_docs
print(f"📄 Total: {len(docs)} documentos listos para procesar")


📄 Cargados 4 documentos web
📄 Cargados 4 documentos PDF
📄 Total: 8 documentos listos para procesar


## 3. DIVISIÓN EN FRAGMENTOS (CHUNKS)

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Divide documentos largos en chunks que quepan en el contexto del LLM.
# chunk_size=1000   → ~750 tokens por chunk (margen de seguridad)
# chunk_overlap=200 → los chunks se solapan 200 chars para no perder
#                     contexto en los cortes (ej: una frase a caballo)
# separators        → corta preferentemente en párrafos, luego líneas,
#                     luego frases; nunca a mitad de palabra
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?"]
)
splits = text_splitter.split_documents(docs)
print(f"✂️ Divididos en {len(splits)} chunks")
print(f"Ejemplo chunk: {len(splits[0].page_content)} chars\n{splits[0].page_content[:200]}...")

✂️ Divididos en 73 chunks
Ejemplo chunk: 620 chars
Trump dice ahora que no descarta desplegar soldados en Irán: "Me dan igual las encuestas" | Internacional

Es noticiaGuerra IránÚltimas noticiasSenegal Estrecho OrmuzLeo piel mariposaPedro SánchezIber...


## 4. EMBEDDINGS + VECTOR STORE

In [42]:
from langchain_huggingface import HuggingFaceEmbeddings

# paraphrase-multilingual-MiniLM-L12-v2: obligatorio cuando mezclas idiomas.
# Español e inglés comparten el mismo espacio vectorial, así las distancias
# entre chunks son comparables independientemente del idioma del documento.
# ⚠️ Si tenías un chroma_db generado con otro modelo, bórralo antes de ejecutar:
#    import shutil; shutil.rmtree('./chroma_db', ignore_errors=True)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Embeddings multilingües listos", embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6875.12it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings multilingües listos model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [43]:
from langchain_community.vectorstores import FAISS

# ─── FAISS (en memoria) ───────────────────────────────────────────────────────
# FAISS convierte cada chunk en un vector de 384 dimensiones y los indexa
# para hacer búsqueda por similitud coseno en ~1ms independientemente del número de chunks.
# Ventajas:  muy rápido, ideal para prototipos y pruebas locales
# Desventaja: no es persistente — se pierde al reiniciar el kernel
#             (por eso lo guardamos en disco en la sección 7)
# Para producción multi-usuario considera Chroma (local) o Pinecone (nube).
vectorstore_faiss = FAISS.from_documents(documents=splits, embedding=embeddings)
print("✅ Vectorstore FAISS listo (384-dim vectores indexados)")
print(f"   Chunks indexados: {vectorstore_faiss.index.ntotal}")


✅ Vectorstore FAISS listo (384-dim vectores indexados)
   Chunks indexados: 73


In [44]:
from langchain_chroma import Chroma
from collections import Counter
import hashlib
# ─── CHROMA (persistente en disco) ───────────────────────────────────────────
# A diferencia de FAISS, Chroma guarda los vectores en disco en ./chroma_db
# y los mantiene entre sesiones sin necesidad de re-embeber los documentos.
#
# Estrategia de inserción con upsert seguro:
# Usamos el hash MD5 del contenido como ID. Antes de insertar comprobamos qué IDs
# ya existen en Chroma y solo insertamos los chunks nuevos.
# Esto evita el DuplicateIDError que lanza chromadb si intentas insertar
# un ID que ya existe en la misma llamada.
#
# ⚠️ Si cambias el modelo de embeddings, borra manualmente chroma_db antes de reejecutar
#    porque los vectores son incompatibles entre modelos distintos:
#    - Desde terminal Windows: rmdir /s /q chroma_db
#    - Desde Python (ejecutar UNA sola vez y luego borrar la celda):
#      import shutil; shutil.rmtree('./chroma_db', ignore_errors=True)
vectorstore_chroma = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)
# Calcular IDs de los chunks actuales
ids_nuevos = [hashlib.md5(doc.page_content.encode('utf-8')).hexdigest() for doc in splits]
# Obtener IDs que ya están en Chroma para no volver a insertarlos
ids_existentes = set(vectorstore_chroma._collection.get()['ids'])
# Filtrar solo chunks nuevos — docs e ids juntos para evitar desincronización
# (filtrarlos por separado puede desalinear ambas listas si hay duplicados entre sí)
pares_nuevos = [
    (doc, id_)
    for doc, id_ in zip(splits, ids_nuevos)
    if id_ not in ids_existentes
]
docs_a_insertar = [doc for doc, _ in pares_nuevos]
ids_a_insertar  = [id_ for _, id_ in pares_nuevos]
if docs_a_insertar:
    vectorstore_chroma.add_documents(documents=docs_a_insertar, ids=ids_a_insertar)
    print(f"✅ Insertados {len(docs_a_insertar)} chunks nuevos")
else:
    print("✅ Chroma ya estaba actualizado, no hay chunks nuevos")
print(f"   Chunks total en Chroma: {vectorstore_chroma._collection.count()}")
# Desglose por fuente — útil para detectar desbalanceo entre web y PDF.
# Si hay muchos más chunks web que PDF, el LLM tenderá a ignorar el PDF
# porque los chunks web dominarán los resultados de búsqueda por similitud.
all_metas = vectorstore_chroma._collection.get(include=['metadatas'])['metadatas']
counts = Counter(m.get('source_type', 'unknown') for m in all_metas)
print("   Desglose por tipo de fuente:", dict(counts))

✅ Chroma ya estaba actualizado, no hay chunks nuevos
   Chunks total en Chroma: 73
   Desglose por tipo de fuente: {'web': 59, 'pdf': 14}


## 5. LLM + CADENA RAG

In [45]:
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate

# ─── CADENA RAG ───────────────────────────────────────────────────────────────
# La cadena RAG conecta tres componentes:
#   1. Retriever → busca los k chunks más similares a la pregunta en el vectorstore
#   2. Prompt    → inyecta los chunks como contexto junto a la pregunta
#   3. LLM       → genera la respuesta usando SOLO el contexto proporcionado
#
# chain_type='stuff' → concatena todos los chunks en un solo bloque de contexto.
#   Es el más simple y funciona bien cuando k es pequeño (4-6 chunks).
#   Para k grande considera 'map_reduce' o 'refine' que procesan chunks por separado.
#
# temperature=0.1 → respuestas más deterministas y precisas.
#   0.0 = siempre la misma respuesta, 1.0 = más creativo pero menos preciso.
#   Para RAG factual se recomienda valores bajos (0.0-0.2).
def llm_rag_chain(vectorstore):
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.1
    )

    # El prompt instruye al LLM a no inventar información fuera del contexto.
    # {context} se sustituye por los chunks recuperados.
    # {question} se sustituye por la pregunta del usuario.
    prompt_template = """Usa SOLO información de los documentos proporcionados.
    Si no encuentras respuesta, di 'No tengo información suficiente'.

    CONTEXTO:
    {context}

    PREGUNTA: {question}
    RESPUESTA CONCISA:"""
    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 4}  # recupera los 4 chunks más similares
        ),
        chain_type_kwargs={"prompt": PROMPT},
        return_source_documents=True  # devuelve los chunks usados para generar la respuesta
    )
    print("🔥 Cadena RAG lista!")
    return qa_chain


In [46]:
qa_chain_faiss = llm_rag_chain(vectorstore_faiss)

🔥 Cadena RAG lista!


In [47]:
qa_chain_chroma = llm_rag_chain(vectorstore_chroma)

🔥 Cadena RAG lista!


## 5.5 RERANKING CON COHERE

In [48]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

# ─── RERANKING CON COHERE ─────────────────────────────────────────────────────
# El retriever básico recupera los k chunks más similares por similitud coseno.
# La similitud coseno mide proximidad semántica general entre vectores, pero no
# necesariamente relevancia exacta para la pregunta concreta. Por ejemplo, un chunk
# que habla de 'Trump' en general puede tener alta similitud coseno con '¿Qué piensa
# Trump sobre Irán?' aunque no mencione Irán en absoluto.
#
# El reranker de Cohere añade un segundo paso de evaluación más preciso:
#   1. Retriever recupera k=10 chunks candidatos (red amplia)
#   2. Cohere evalúa cada chunk contra la pregunta usando un modelo de cross-encoding
#      (lee pregunta + chunk juntos, no como vectores separados) → puntuación de relevancia
#   3. Reordena los 10 chunks por esa puntuación y devuelve solo los top_n=4 mejores al LLM
#
# Resultado: el LLM recibe los 4 chunks más relevantes en lugar de los 4 más
# similares semánticamente → respuestas más precisas, especialmente en preguntas
# específicas donde la similitud coseno no es suficiente.
#
# Requiere COHERE_API_KEY en el archivo .env → https://dashboard.cohere.com/api-keys
compressor = CohereRerank(
    model="rerank-multilingual-v3.0",  # modelo multilingüe, compatible con español e inglés
    top_n=4                             # devuelve los 4 chunks más relevantes al LLM
)

# ContextualCompressionRetriever envuelve el retriever base con el reranker:
# primero recupera k=10 candidatos con similitud coseno, luego Cohere los reordena
# y filtra a top_n=4. El LLM solo ve esos 4 chunks finales.
retriever_rerank_faiss = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore_faiss.as_retriever(search_kwargs={'k': 10})
)
retriever_rerank_chroma = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore_chroma.as_retriever(search_kwargs={'k': 10})
)

# Crear cadenas RAG con reranking sustituyendo el retriever base por el reranker
qa_chain_faiss_rerank  = llm_rag_chain(vectorstore_faiss)
qa_chain_chroma_rerank = llm_rag_chain(vectorstore_chroma)
qa_chain_faiss_rerank.retriever  = retriever_rerank_faiss
qa_chain_chroma_rerank.retriever = retriever_rerank_chroma

print("✅ Cadenas RAG con reranking listas")


🔥 Cadena RAG lista!
🔥 Cadena RAG lista!
✅ Cadenas RAG con reranking listas


## 5.6 PRUEBA DE RERANKING

In [49]:
# ─── COMPARACIÓN VISUAL SIN vs CON RERANKING ─────────────────────────────────
# Muestra los chunks que recupera cada retriever para la misma pregunta.
# Permite ver cualitativamente si el reranking mejora la selección de chunks.
def comparar_reranking(pregunta, vectorstore, retriever_rerank):
    print(f'Pregunta: {pregunta}')
    print('\n--- SIN reranking (similitud coseno) ---')
    chunks_base = vectorstore.as_retriever(search_kwargs={'k': 4}).invoke(pregunta)
    for i, doc in enumerate(chunks_base):
        print(f'{i+1}. [{doc.metadata.get("source_type", "?").upper()}] {doc.page_content[:120]}...')
    print('\n--- CON reranking (Cohere) ---')
    chunks_rerank = retriever_rerank.invoke(pregunta)
    for i, doc in enumerate(chunks_rerank):
        print(f'{i+1}. [{doc.metadata.get("source_type", "?").upper()}] {doc.page_content[:120]}...')
    return chunks_base, chunks_rerank

chunks_base, chunks_rerank = comparar_reranking(
    'Qué líder supremo murió?',
    vectorstore_chroma,
    retriever_rerank_chroma
)

# ─── MÉTRICAS ─────────────────────────────────────────────────────────────────
# MRR (Mean Reciprocal Rank): mide en qué posición aparece el primer chunk relevante.
#   Posición 1 → 1.0 | Posición 2 → 0.5 | Posición 3 → 0.33 | Ninguno → 0.0
#   Cuanto más cercano a 1.0, mejor.
#   Limitación: solo mide el primer chunk relevante, no la calidad del resto.
#
# Precisión: mide qué porcentaje de los 4 chunks contienen contenido relevante.
#   1.0 = todos relevantes | 0.5 = la mitad | 0.0 = ninguno relevante
#   Complementa al MRR capturando la calidad global del conjunto de chunks.
#
# Ambas métricas definen relevancia por la presencia de keywords en el chunk.
# Para evaluación más rigurosa con preguntas/respuestas esperadas → LangSmith.
#
# ⚠️ Las keywords deben ser específicas del tema de la pregunta, no del corpus general.
#    En este caso: keywords = ['Alí', 'Alí Jameneí', 'líder supremo', 'muerte', 'falleció', 'murió']
#    para la pregunta "¿Qué líder supremo murió?"
#
#    Resultado observado:
#      - Sin reranking: MRR=1.00, Precisión=0.75 → recupera 3 chunks relevantes pero
#        incluye ruido (ej. chunk sobre Marcos Ondarra, irrelevante para la pregunta).
#      - Con reranking (Cohere): MRR=1.00, Precisión=0.50 → sube al top los chunks
#        que mencionan directamente a Jameneí como líder supremo muerto, pero es más
#        estricto y descarta chunks que solo rozan el tema.
#
#    Conclusión: el MRR empata (el chunk más relevante aparece primero en ambos casos),
#    pero el reranking demuestra su valor eliminando ruido y priorizando chunks
#    directamente relevantes para la pregunta concreta, a costa de menor precisión
#    global. En producción, esto se traduce en mejor contexto para el LLM.

keywords = ['Alí', 'Alí Jameneí', 'líder supremo', 'muerte', 'falleció', 'murió']

def mrr(chunks, keywords):
    if isinstance(keywords, str):
        keywords = [keywords]
    for i, doc in enumerate(chunks):
        if any(kw.lower() in doc.page_content.lower() for kw in keywords):
            return 1 / (i + 1)
    return 0.0

def precision(chunks, keywords):
    if isinstance(keywords, str):
        keywords = [keywords]
    relevantes = sum(
        1 for doc in chunks
        if any(kw.lower() in doc.page_content.lower() for kw in keywords)
    )
    return relevantes / len(chunks) if chunks else 0.0

mrr_base    = mrr(chunks_base, keywords)
mrr_rerank  = mrr(chunks_rerank, keywords)
prec_base   = precision(chunks_base, keywords)
prec_rerank = precision(chunks_rerank, keywords)

print(f'\n{"":25} {"Sin reranking":>15} {"Con reranking":>15}')
print(f'{"─"*55}')
print(f'{"MRR":25} {mrr_base:>15.2f} {mrr_rerank:>15.2f}')
print(f'{"Precisión":25} {prec_base:>15.2f} {prec_rerank:>15.2f}')


Pregunta: Qué líder supremo murió?

--- SIN reranking (similitud coseno) ---
1. [WEB] . Eufórico y con el patriotismo exaltado en el año de celebración del 250 aniversario de la independencia, ha hecho tamb...
2. [WEB] "El ataque fue tan exitoso que acabamos con la mayoría de candidatos", aseguró Trump en la llamada con ABC News.El presi...
3. [WEB] Trump afirma que los candidatos para reemplazar al ayatolá Jameneí están todos muertos tras los bombardeosACTUALIDAD "El...
4. [WEB] Trump afirma que los candidatos para reemplazar al ayatolá Jameneí están todos muertos tras los bombardeos

Hoy interesa...

--- CON reranking (Cohere) ---
1. [WEB] "El ataque fue tan exitoso que acabamos con la mayoría de candidatos", aseguró Trump en la llamada con ABC News.El presi...
2. [WEB] Trump afirma que los candidatos para reemplazar al ayatolá Jameneí están todos muertos tras los bombardeosACTUALIDAD "El...
3. [WEB] El mandatario ha celebrado la potencia del Ejército estadounidense —«el mejor del mu

## 6. ¡PRUEBAS RAG! 🔥

In [52]:
def rag_test(qa_chain, text_query):
    resultado = qa_chain.invoke({"query": text_query})
    print("🤖 RESPUESTA:")
    print(resultado["result"])
    print("\n" + "="*60)
    print("📄 FUENTES (top-4):")
    for i, doc in enumerate(resultado["source_documents"][:4]):
        print(f"{i+1}. [{doc.metadata.get('source_type', '?').upper()}] {doc.metadata.get('source', doc.metadata.get('file_path', 'Desconocido'))[:60]}")
        print(f"   Preview: {doc.page_content[:150]}...\n")


In [53]:
consulta = "¿Me puedes hablar de los productos organicos y alguna irregularidad que se mencione y se realice?"
rag_test(qa_chain_faiss, consulta)

🤖 RESPUESTA:
Según la información proporcionada, los productos orgánicos deben cumplir con ciertas regulaciones y requisitos de etiquetado. Algunas irregularidades que se mencionan incluyen:

* No identificar separadamente los ingredientes orgánicos y no orgánicos en la declaración de ingredientes.
* Utilizar el sello de USDA orgánico en productos que no son 100% orgánicos.
* Representar un producto como orgánico cuando no cumple con los requisitos de etiquetado orgánico.
* No incluir la declaración "certificado orgánico por" en la etiqueta.
* No cumplir con los requisitos de tamaño y formato de la declaración "hecho con ingredientes orgánicos" y el porcentaje de ingredientes orgánicos.

Es importante destacar que los productos que se etiquetan como "hecho con ingredientes orgánicos" deben cumplir con ciertos requisitos, como que al menos el 70% de los ingredientes sean orgánicos y que se identifiquen separadamente los ingredientes orgánicos y no orgánicos en la declaración de ingredie

In [54]:
consulta = "What does Trump think about Iran?"
rag_test(qa_chain_chroma, consulta)

🤖 RESPUESTA:
Trump no descarta el envío de tropas a Irán "si son necesarias" y afirma que "los estamos destrozando". También menciona que "la gran ola" de ataques contra Irán "está a punto de llegar". Esto sugiere que Trump tiene una postura agresiva y crítica hacia Irán.

📄 FUENTES (top-4):
1. [WEB] https://theobjective.com/internacional/2026-03-02/trump-envi
   Preview: Internacional

 
 Trump no descarta el envío de tropas a Irán «si son necesarias»: «Los estamos destrozando» 
El presidente de EEUU anuncia que la «gr...

2. [WEB] https://www.elperiodico.com/es/internacional/20260224/trump-
   Preview: DirectoIsrael asegura ahora que el conflicto en el Líbano es un "frente principal" de guerra de Irán Intervención ante las CámarasTrump amenaza a Irán...

3. [WEB] https://www.elmundo.es/internacional/2026/03/02/69a5b73ce85e
   Preview: Estás en: Internacional Europa América Asia África Oceanía Más SUSCRÍBETE 20% DTO. INTERNACIONALEEUUTrump dice ahora que no descarta desplegar soldado.

## 7. GUARDAR/RECARGAR VECTORSTORE (PRODUCCIÓN)

In [55]:
# ─── PERSISTENCIA FAISS ──────────────────────────────────────────────────────
# FAISS no es persistente por defecto — al reiniciar el kernel se pierde el índice
# y habría que re-embeber todos los documentos (lento con modelos grandes).
# save_local genera dos ficheros en disco:
#   - index.faiss → los vectores
#   - index.pkl   → los metadatos (textos, fuentes, etc.)
vectorstore_faiss.save_local("mi_rag_index")

# Al recargar, allow_dangerous_deserialization=True es obligatorio porque
# FAISS usa pickle para los metadatos, que puede ejecutar código arbitrario.
# Solo carga índices que hayas generado tú mismo; nunca de fuentes externas.
vectorstore_reloaded = FAISS.load_local(
    "mi_rag_index", embeddings, allow_dangerous_deserialization=True
)
print("💾 FAISS guardado y recargado correctamente")
print(f"   Chunks en el índice: {vectorstore_reloaded.index.ntotal}")


💾 FAISS guardado y recargado correctamente
   Chunks en el índice: 73


In [56]:
# ─── PERSISTENCIA CHROMA ─────────────────────────────────────────────────────
# Chroma guarda automáticamente en ./chroma_db cada vez que añades documentos.
# Para recargar en una nueva sesión basta con crear el objeto Chroma con el mismo
# persist_directory — no hace falta volver a insertar los documentos.
#
# ⚠️ No ejecutes add_documents después de recargar sin comprobar qué IDs ya existen
#    (la celda de Chroma de la sección 4 ya lo hace correctamente con el filtro de IDs existentes).
# vectorstore_chroma_reloaded = Chroma(
#     persist_directory="./chroma_db",
#     embedding_function=embeddings
# )
# print("💾 Chroma recargado (ya persistente)")
# print(f"   Chunks: {vectorstore_chroma_reloaded._collection.count()}")


## 🎯 PRÓXIMOS PASOS

| Mejora | Cómo | Beneficio | Estado |
|--------|------|-----------|--------|
| **Streaming** | `qa_chain.stream()` | Respuestas token a token en tiempo real | - |
| **Reranking** | `CohereRerank` | Reordena los chunks recuperados → precisión +30% | ✅ |
| **Cloud DB** | `Chroma` / `Pinecone` | Persistencia escalable y multi-usuario | ✅ |
| **PDFs** | `PyMuPDFLoader` | Ingesta documentos empresariales | ✅ |
| **Multi-idioma** | `paraphrase-multilingual-MiniLM-L12-v2` | Embeddings nativos en español y otros idiomas | ✅ |
| **Evaluación con LangSmith** | `langsmith` + `evaluate()` | Mide la calidad de la cadena RAG | - |
